# 🛠️ AI Text Processor & 🎙️ TTS Audio Book Generator

This delightful tool uses Kokoro TTS and brilliant AI models to spin your ideas into custom audiobooks right in Google Colab—no technical wizardry required!

**Any Input**: Paste text, upload a .txt file, or give the Vision AI an image to describe!

**Smart Processing**: Clean up messy text, use custom prompts, or use the "Generate" & "Recursion" tools to loop a tiny concept into a sprawling story.

**Choose Your Processor**: Pick from strict text-formatting models or wildly creative storytelling ones.

**Beautiful Voices**: Turn your final text into a seamless, high-quality audiobook with a wide selection of voices.

**Safe & Sound**: Auto-save your masterpieces to Google Drive with optional password encryption!

*A Tiny Note*: The AI loves to make text flow smoothly for audio, so it might slightly tweak your words. Because of these charming quirks, please avoid using this for strict math or highly technical documents where format and punctuation is critical!

In [ ]:
# @title 🛠️ AI Text Processor & 🎙️ TTS (Text to Speech) Generator 🛠️
# @markdown ### Select your Task and Input Source below:
Task = "Both: Process Text then Generate TTS" # @param ["Text Processor Only", "TTS Generator Only", "Both: Process Text then Generate TTS"]
input_source = "Upload File (.txt or Image)" # @param ["Text Box", "Upload File (.txt or Image)"]
# @markdown **image_prompt:** *(Optional)* If uploading an image, add specific context for thingast can't be "Seen" (e.g., "The person in the red dress is named Sarah").
image_prompt = "" # @param {type:"string"}
# @markdown **text:** *(Optional)* If input_source is 'Text Box', paste what you wan to be processed here.
text = "" # @param {type:"string"}
# @markdown <hr />

# @markdown ### ⚙️ General Settings:
# @markdown **save_to_google_drive:** Automatically save outputs to Google Drive? (Requires login)
save_to_google_drive = False # @param {type:"boolean"}
# @markdown **zip_password:** If saving to Drive, encrypt the file(s) with this password. (Leave blank for no encryption)
zip_password = "" # @param {type:"string"}
# @markdown **output_filename:** If using 'Text Box' input name your files here. This uses input filenames if you give it a file.
output_filename = "Processed_File" # @param {type:"string"}
# @markdown <hr />

# @markdown ### 🎙️ TTS Settings (Text to Speech) Settings:
# @markdown **voice:** Select the voice you wish to use. All valid entities and samples can be found [HERE](https://huggingface.co/onnx-community/Kokoro-82M-v1.0-ONNX#voicessamples) in the `Voices/Samples` section.<br />
# @markdown *Note:* Voice format is a/b (American/British) f/m (Feminine/Masculine) _name, E.G. af_nova = An American, Feminine voice.
voice = "af_nova" # @param ["af_nova", "af_heart", "bf_emma", "am_fenrir", "bm_daniel"] {allow-input: true}
# @markdown <hr />

# @markdown ### 🛠️ T2T (Text to Text) Settings:
# @markdown This is the instruction that the "Text processor" uses to process the file. <br />
# @markdown **TextCleaning**: Keeps the current text and removes formatting, page numbers, etc. to make it ready for TTS. E.G. You have copied data out of a PDF and it is awfully formatted. <br />
# @markdown **TextGeneration**: Takes your input and expands it to be more in-depth. E.G. taking "A story about a cat" and turning it into a full story. <br />
# @markdown **Custom**: Use the text box here to give the "Text to Text" engine your own instructions!
prompt_type = "TextGeneration" # @param ["TextCleaning", "TextGeneration", "Custom"]
custom_prompt = "" # @param {type:"string"}

# --- Dynamic Temperature Setting ---
if prompt_type == "TextCleaning":
    gen_temp = 0.1 # Low temp for strict adherence and formatting
else:
    gen_temp = 0.75 # Higher temp for creativity in storytelling

if prompt_type == "TextCleaning":
    system_prompt = (
        "You are an expert audio-text preparer. Your task is to process this text "
        "so it reads smoothly for Text-to-Speech processing. 1. Remove random line breaks "
        "to reconstruct proper flowing paragraphs. 2. Fix broken hyphenations (e.g., "
        "'para- graph' becomes 'paragraph'). 3. Normalize spacing by removing extra spaces "
        "or tabs. 4. Delete inline headers, footers, page numbers, and stray isolated numbers. "
        "5. DO NOT rewrite, summarize, or change the author's original words. Output ONLY the "
        "processed text with no conversational filler."
    )
elif prompt_type == "TextGeneration":
    system_prompt = (
        "You are an award-winning novelist and master storyteller. Your task is to write a compelling, "
        "deeply immersive story based on the provided text or concept. "
        "1. Structure: Build a complete narrative arc with a captivating hook, escalating tension, "
        "a distinct climax, and a resonant resolution. "
        "2. World & Character: Craft multi-dimensional characters with distinct voices and internal "
        "motivations. Anchor them in a vivid, lived-in setting using visceral sensory details. Apply the "
        "'show, don't tell' principle. "
        "3. Pacing & Depth: Expand the core concept substantially to ensure a lengthy, detailed read. "
        "Use varied sentence structures to control the pacing naturally. "
        "4. Tone: Establish a consistent atmosphere that aligns perfectly with the input's genre, "
        "prioritizing emotional authenticity. "
        "5. Constraints: DO NOT include titles, introductions, meta-commentary, or conversational filler. "
        "Output absolutely nothing but the story itself."
    )
elif prompt_type == "Custom":
    system_prompt = custom_prompt

# @markdown **recursion_loops:** How many times should the AI take its own output and feed it back into itself to expand/continue the text? (1 = process input once. 2+ = continue extending the text, retaining the WHOLE story as context). <br />
# @markdown **Note**: Leave this as "1" for TextCleaning.
recursion_loops = 1 # @param {type:"slider", min:1, max:20, step:1}

# @markdown **repetition_penalty:** Helps prevent the AI from looping or repeating phrases (like starting every sentence with "And so"). 1.0 is no penalty, 1.15 is a good default.
repetition_penalty = 1.15 # @param {type:"slider", min:1.0, max:2.0, step:0.05}

# @markdown ### Text Processing Model Selector:
# @markdown **model_choice:** Select which AI model to use for the text processing. <br />
# @markdown * `Qwen/Qwen2.5-3B-Instruct` - Strict & Precise. Excellent at standard text formatting, logical processing, and exact instruction following. <br />
# @markdown * `dphn/Dolphin3.0-Qwen2.5-3b` - Highly Compliant Qwen. Delivers the same precise formatting capabilities, but is tuned for absolute adherence to your prompt without hesitation. <br />
# @markdown * `dphn/Dolphin3.0-Llama3.2-3B` - The Creative Writer, this excels at natural prose, storytelling, and narrative expansion.
model_choice = "Qwen/Qwen2.5-3B-Instruct" # @param ["Qwen/Qwen2.5-3B-Instruct", "dphn/Dolphin3.0-Qwen2.5-3b", "dphn/Dolphin3.0-Llama3.2-3B"] {allow-input: true}

import os
import sys
import subprocess
import shutil
import re
import urllib.request
import numpy as np
from IPython.display import Audio, display, clear_output
from google.colab import files

# --- BYPASS HUGGING FACE TOKEN POPUP ---
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TOKEN_WARNING"] = "1"

# --- MOUNT GOOGLE DRIVE FIRST ---
if save_to_google_drive:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        print("📂 Mounting Google Drive, Please login using the popup window.")
        drive.mount('/content/drive')

# --- ENSURE 7-ZIP IS INSTALLED ---
if save_to_google_drive and zip_password.strip():
    try:
        subprocess.run(["7z"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except FileNotFoundError:
        print("📦 Installing 7-zip for encryption...")
        subprocess.run("sudo DEBIAN_FRONTEND=noninteractive apt-get update -qq && sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq p7zip-full", shell=True, check=True)

# --- 1. GET THE INPUT TEXT (OR IMAGE) ---
raw_input = ""
base_name = output_filename

if input_source == "Upload File (.txt or Image)":
    print("📂 Awaiting file upload... Please select your file (.txt or Image) below.")
    uploaded = files.upload()
    if not uploaded:
        print("❌ No file uploaded. Execution stopped.")
        sys.exit()

    original_filename = list(uploaded.keys())[0]
    base_name = os.path.splitext(original_filename)[0]
    ext = os.path.splitext(original_filename)[1].lower()

    image_extensions = ['.png', '.jpg', '.jpeg', '.webp', '.bmp', '.gif', '.tiff']

    if ext in image_extensions:
        print(f"🖼️ Detected image file '{original_filename}'. Analyzing with Vision model...")

        try:
            from PIL import Image
            import torch
            from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration
        except ImportError:
            print("📦 Installing required image processing libraries... (Silently)")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "Pillow", "torch", "accelerate"], check=True, capture_output=True)
            from PIL import Image
            import torch
            from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration

        image = Image.open(original_filename).convert('RGB')
        device_name = "cuda" if torch.cuda.is_available() else "cpu"
        print("⏳ Loading Vision Model (LLaVA-OneVision-Qwen2-0.5B)...")

        model_id = "llava-hf/llava-onevision-qwen2-0.5b-ov-hf"
        processor = AutoProcessor.from_pretrained(model_id)
        vision_model = LlavaOnevisionForConditionalGeneration.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True
        ).to(device_name)

        print("👁️ Extracting visual details...")
        if image_prompt.strip():
            combined_prompt = f"The subject of this image is: {image_prompt.strip()}. Describe this image in detail, capturing all visual elements."
        else:
            combined_prompt = "Describe this image in detail. Be thorough and capture all the visual elements."

        conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": combined_prompt}]}]

        prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
        inputs = processor(images=image, text=prompt, return_tensors="pt").to(device_name, torch.float16)
        out = vision_model.generate(**inputs, max_new_tokens=300)

        generated_ids = out[0][inputs.input_ids.shape[1]:]
        description = processor.decode(generated_ids, skip_special_tokens=True).strip()

        if image_prompt.strip():
            raw_input = f"Context: {image_prompt.strip()}\n\nImage Description:\n{description}"
        else:
            raw_input = description

        print(f"📝 Image Description generated:\n{raw_input}")

        del vision_model
        del processor
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    else:
        try:
            raw_input = uploaded[original_filename].decode('utf-8')
            print(f"✅ Loaded text file '{original_filename}' successfully.")
        except UnicodeDecodeError:
            try:
                raw_input = uploaded[original_filename].decode('latin-1')
                print(f"✅ Loaded text file '{original_filename}' successfully (latin-1 encoding).")
            except Exception:
                print(f"❌ Error: Could not decode text file '{original_filename}'.")
                sys.exit()

else:
    raw_input = text

if not raw_input.strip():
    print("⚠️ No input detected. Please provide text or upload a file.")
    sys.exit()

current_text = raw_input

# --- 2. TEXT PROCESSOR LOGIC ---
if "Text Processor" in Task or "Both" in Task:
    print("\n" + "="*50)
    print("🖨️ STARTING AI TEXT PROCESSOR...")
    print("="*50)

    try:
        import transformers
    except ImportError:
        print("📦 Installing required libraries... (Silently)")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate", "torch"], check=True, capture_output=True)

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if 'model' in globals() and globals().get('current_model_name') != model_choice:
        print("🧹 Unloading previous model to free up VRAM...")
        del globals()['model']
        del globals()['tokenizer']
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if 'model' not in globals() or 'tokenizer' not in globals():
        print(f"⏳ Loading {model_choice} into GPU... (Takes 2-3 minutes)")
        tokenizer = AutoTokenizer.from_pretrained(model_choice)
        model = AutoModelForCausalLM.from_pretrained(
            model_choice,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        current_model_name = model_choice
        print("✅ Model loaded successfully!")
    else:
        print(f"⚡ Model ({model_choice}) already in memory. Skipping setup.")

    def process_text_with_ai(messages):
        formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer([formatted_prompt], return_tensors="pt").to(model.device)
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=2000,
            temperature=gen_temp,
            repetition_penalty=repetition_penalty,
            do_sample=True,
        )
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    final_txt_filename = f"{base_name}_processed.txt"
    print(f"✨ Starting AI processing... Output will be saved to: {final_txt_filename}")

    chunk_size = 2500
    words = current_text.replace('\n', ' \n ').split(' ')
    chunks = []
    current_chunk = ""

    for word in words:
        if len(current_chunk) + len(word) + 1 < chunk_size:
            current_chunk += word + " "
        else:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
            current_chunk = word + " "
    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    print(f"🧩 Document safely split into {len(chunks)} manageable chunks.")
    if len(chunks) > 1 and prompt_type == "TextGeneration":
        print("⚠️ Note: Because the input is long, generation will happen in separate chunks.")

    with open(final_txt_filename, "w", encoding="utf-8") as f:
        f.write("")

    full_generated_story = ""

    for i, chunk in enumerate(chunks):
        print(f"⏳ Processing section {i+1} of {len(chunks)}...")
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Please process the following text:\n\n{chunk}"}
        ]
        processed_chunk = process_text_with_ai(messages)
        full_generated_story += processed_chunk + "\n\n"

        with open(final_txt_filename, "a", encoding="utf-8") as f:
            f.write(processed_chunk + "\n\n")
            f.flush()
            os.fsync(f.fileno())

    if recursion_loops > 1:
        print(f"\n🔄 Starting recursion to expand the text ({recursion_loops - 1} additional passes)...")

        for loop in range(recursion_loops - 1):
            print(f"⏳ Recursion loop {loop+1} of {recursion_loops - 1}...")

            recursion_messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Here is the text generated so far:\n\n{full_generated_story.strip()}\n\nPlease continue the story/text from exactly where it left off. Maintain the same style, tone, and formatting. Expand upon the narrative. Do not repeat what was already written. Output ONLY the new continuation without any commentary."}
            ]

            continuation = process_text_with_ai(recursion_messages)
            full_generated_story += continuation + "\n\n"

            with open(final_txt_filename, "a", encoding="utf-8") as f:
                f.write(continuation + "\n\n")
                f.flush()
                os.fsync(f.fileno())

    print(f"🎉 Done! Processed text completely, saved to: {final_txt_filename}")

    with open(final_txt_filename, "r", encoding="utf-8") as f:
        current_text = f.read()

    if save_to_google_drive:
        if zip_password.strip():
            archive_name = f"{final_txt_filename}.7z"
            print(f"🔒 Encrypting {final_txt_filename} (with hidden filenames)...")
            subprocess.run(["7z", "a", f"-p{zip_password}", "-mhe=on", archive_name, final_txt_filename], stdout=subprocess.DEVNULL)
            drive_path = f"/content/drive/MyDrive/{archive_name}"
            shutil.copy(archive_name, drive_path)
        else:
            drive_path = f"/content/drive/MyDrive/{final_txt_filename}"
            shutil.copy(final_txt_filename, drive_path)

        print(f"💾 Successfully saved text to Google Drive at: {drive_path}")

    try:
        if "Both" in Task:
            print(f"⚠️ Text processing finished. The text file download will be triggered at the very end.")
        else:
            files.download(final_txt_filename)
    except Exception as e:
        print(f"⚠️ Could not trigger automatic download. Find '{final_txt_filename}' on the left menu.")

# --- 3. TTS GENERATOR LOGIC ---
if "TTS Generator" in Task or "Both" in Task:
    print("\n" + "="*50)
    print("🎙️ STARTING KOKORO TTS GENERATOR (ONNX)...")
    print("="*50)

    import gc
    try:
        import torch
        for var in ['model', 'tokenizer', 'vision_model', 'processor']:
            if var in globals():
                del globals()[var]
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

    try:
        from kokoro_onnx import Kokoro
        import soundfile as sf
    except ImportError:
        print("📦 Installing Python 3.13 compatible Kokoro ONNX engine...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "kokoro-onnx", "soundfile"],
            check=True
        )
        from kokoro_onnx import Kokoro
        import soundfile as sf
        print("✅ ONNX dependencies installed successfully!")

    model_file = "kokoro-v1.0.onnx"
    voices_file = "voices-v1.0.bin"

    if not os.path.exists(model_file):
        print("📥 Downloading Kokoro ONNX model (~300MB)...")
        urllib.request.urlretrieve(
            "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx",
            model_file
        )

    if not os.path.exists(voices_file):
        print("📥 Downloading Kokoro voice profiles (~15MB)...")
        urllib.request.urlretrieve(
            "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin",
            voices_file
        )

    final_wav_filename = f"{base_name}_processed.wav" if "Both" in Task else f"{base_name}.wav"

    print("\nInitializing Kokoro ONNX Engine...")
    kokoro = Kokoro(model_file, voices_file)

    lang_map = {'a': 'en-us', 'b': 'en-gb', 'f': 'fr-fr', 'e': 'es', 'j': 'ja', 'z': 'zh'}
    lang = lang_map.get(voice[0], 'en-us')

    print(f"Generating speech for voice: {voice}...")

    text_chunks = [chunk.strip() for chunk in re.split(r'(?<=[.!?\n])\s+', current_text) if chunk.strip()]
    total_chunks = len(text_chunks)
    audio_chunks = []
    sample_rate = 24000

    for i, chunk_text in enumerate(text_chunks):
        samples, sample_rate = kokoro.create(chunk_text, voice=voice, speed=1.0, lang=lang)
        audio_chunks.append(samples)
        print(f"  -> Processed chunk {i+1} out of {total_chunks}...")

    if audio_chunks:
        print("Merging chunks and saving file...")
        final_audio = np.concatenate(audio_chunks)

        sf.write(final_wav_filename, final_audio, sample_rate)
        print(f"✅ Successfully created: {final_wav_filename}")

        if save_to_google_drive:
            if zip_password.strip():
                archive_name = f"{final_wav_filename}.7z"
                print(f"🔒 Encrypting {final_wav_filename} ...")
                subprocess.run(["7z", "a", f"-p{zip_password}", "-mhe=on", archive_name, final_wav_filename], stdout=subprocess.DEVNULL)
                drive_path = f"/content/drive/MyDrive/{archive_name}"
                shutil.copy(archive_name, drive_path)
            else:
                drive_path = f"/content/drive/MyDrive/{final_wav_filename}"
                shutil.copy(final_wav_filename, drive_path)

            print(f"💾 Successfully saved audio to Google Drive at: {drive_path}")

        display(Audio(final_wav_filename, autoplay=True))

        print("\n📥 Triggering Downloads...")
        if "Both" in Task:
            print("⚠️ NOTE: Your browser may ask for permission to download multiple files at once. Please click 'Allow' in your URL bar if prompted.")
            try:
                files.download(final_txt_filename)
            except Exception as e:
                pass

        try:
            files.download(final_wav_filename)
        except Exception as e:
            print(f"⚠️ Could not trigger automatic download. Find your files on the left menu.")

    else:
        print("❌ Error: Audio generation failed.")